# Clasificar competiciones (qué es cada competición donde jugó cada jugador)

**Idea:** para cada `competition_id` de `silver_player_stats` y `silver_achievements` se busca en Transfermarkt su categoría:
1. **Índices de competiciones** (`/wettbewerbe/{región}`, con todas sus páginas): la tabla agrupa las competiciones bajo subtítulos
   (`First Tier`, `Second Tier`, ..., `Youth league`, `Domestic Cup`, `Domestic Super Cup`, etc.). Se toma el subtítulo de la **tabla**
   (no el de la caja de links rápidos de arriba, que no tiene categorías). Esas competiciones no se visitan.
2. **Página de la competición** (`/x/startseite/wettbewerb/{id}`) para las que no están en la tabla de ningún índice: si el encabezado
   trae `League level`, se guarda; las copas no traen nada en el encabezado y quedan con `estado = sin_nivel`.

**Lee solo `silver_player_stats` y `silver_achievements` (y su propia tabla de salida). No usa ninguna tabla gold**, así que va entre silver y gold, igual que `clasificar_clubes_externos`.

**Tabla de salida:** `workspace.inferiores.bronze_competiciones`

| Columna | Contenido |
|---|---|
| `competition_id` | ID de la competición en Transfermarkt |
| `nombre` | Nombre de la competición (título de su página o, si no hay, el de las tablas silver) |
| `tipo_url` | Adónde terminó la URL: `wettbewerb` o `pokalwettbewerb` |
| `categoria_tm` | Categoría de Transfermarkt: el subtítulo del índice o el campo `League level` |
| `fuente_categoria` | `indice` (subtítulo de la tabla del índice) o `pagina` (campo `League level` del encabezado) |
| `indice` | Región del índice donde apareció (`afrika`, `amerika`, `asien`, `europa`, `europaJugend` o `fifa`). Solo para `fuente_categoria = indice` |
| `url_final` | URL final de la página visitada. Nula para las que salen de un índice |
| `estado` | Ver abajo |
| `visitado_en` | Momento en que se generó la fila |

| `estado` | Significado |
|---|---|
| `indice` | La categoría salió de la tabla de un índice y no se visitó la página |
| `ok` | La página trae `League level` |
| `sin_nivel` | La página no trae nivel: típico de copas |
| `sin_pagina` | La página no devolvió contenido (404 o descarga fallida tras los reintentos) |

`tipo_url` **no es un indicador de tipo**: hay copas y campeonatos estaduales en ambos.

**Orden de ejecución:** CELDA 1 (funciones), CELDA 3 (rondas; se puede cortar y retomar) y CELDA 4 (resumen). No hay CELDA 2.
Si se reinicia la sesión, correr la CELDA 1 de nuevo y seguir.

## Reglas de acceso al sitio que aplica el código

- Descarga `robots.txt` de `www.transfermarkt.com` al inicio y, antes de cada pedido no cacheado, consulta si la ruta está permitida. Si no lo está, lanza `RuntimeError`.
- Espera 5 s (`DELAY`) antes de cada pedido no cacheado.
- Ante HTTP `202`, `403` o `429` espera 120 s y reintenta, hasta 2 veces. Si vuelve a pasar, lanza `RuntimeError`. La celda de rondas atrapa ese error, guarda lo pendiente, imprime `CORTE` y termina sin error.
- Guarda cada página descargada en un Volume de Databricks, junto con su URL final, y la lee de ahí en las corridas siguientes.

## Control del ciclo
`%run ./ciclo` carga las funciones del notebook `ciclo`. La celda siguiente llama a `motivo_no_corre("publicar")`: si el ciclo no está listo para publicar (todavía faltan jugadores o ya se publicó), termina con `SALTEADO: <motivo>` y no hace nada.

In [ ]:
%run ./ciclo

In [ ]:
motivo_salida = motivo_no_corre("publicar")
if motivo_salida:
    dbutils.notebook.exit("SALTEADO: " + motivo_salida)

## CELDA 1: configuración y funciones
Correr primero en cada sesión.

| Elemento | Qué hace |
|---|---|
| `TM`, `UA`, `CACHE`, `DELAY` | Sitio, User-Agent, carpeta de caché en el Volume y espera entre páginas (5 s) |
| `TABLA`, `COLS`, `SCHEMA` | Tabla de salida y su esquema (todas las columnas son texto). La celda crea la tabla si no existe |
| `INDICES` | Las seis regiones de `/wettbewerbe/`: `afrika`, `amerika`, `asien`, `europa`, `europaJugend` y `fifa` |
| `curl(url, out)` | Descarga la URL y devuelve el código HTTP y la URL final, después de seguir las redirecciones |
| `get(url)` | Devuelve `(código, html, url_final)`. Usa el caché, consulta `robots.txt`, espera `DELAY`, reintenta hasta 6 veces y maneja el freno |
| `limpiar(h)` | Convierte HTML en texto plano: saca `script`, `style` y etiquetas, y normaliza los espacios |
| `tabla_items(h)` | Devuelve el HTML de la tabla `items` de la página (la de las competiciones), respetando las tablas anidadas |
| `leer_indices()` | Recorre todas las páginas de los seis índices y devuelve `{competition_id: (región, subtítulo)}` |
| `leer_pagina(html, final)` | De la página de una competición devuelve `(título, tipo_url, nivel)` |
| `ahora()` | Hora actual en formato ISO (igual que la de `ciclo`) |
| `guardar(rows)` | Agrega filas a la tabla de salida |
| `pendientes()` | Competiciones que todavía no están en la tabla de salida, ordenadas por partidos jugados, de mayor a menor |

**Cómo lee `leer_indices()`**

1. Por cada región pide la primera página y detecta cuántas páginas hay a partir de los enlaces `?page=N`.
2. De cada página toma solo la tabla `items`. Si no la encuentra, la cuenta como página sin tabla.
3. Recorre la tabla en orden: una fila de subtítulo (`extrarow`) fija el subtítulo actual (hasta 60 caracteres) y cada enlace a una competición que viene después recibe ese subtítulo.
4. Si una competición aparece en más de un lugar, se queda con el primero y cuenta un conflicto cuando el subtítulo es distinto.
5. Imprime cuántas competiciones encontró, cuántas páginas no tenían tabla y cuántos IDs tuvieron subtítulos distintos.

**Cómo lee `leer_pagina()`**

| Campo | Cómo se obtiene |
|---|---|
| Título | Etiqueta `<title>` sin el sufijo `\| Transfermarkt` |
| `tipo_url` | `pokalwettbewerb` si la URL final contiene `/pokalwettbewerb/`; si no, `wettbewerb` |
| Nivel | Texto que sigue a `League level:`, hasta 45 caracteres y cortado en la primera palabra de corte (`Reigning`, `Record`, `Number`, `UEFA`, `Total`, `Most`, `€` o `-`). Nulo si no está |

**Qué incluye `pendientes()`**

- Las competiciones de `silver_player_stats` con al menos un partido (`appearances > 0`), con la suma de partidos.
- Todas las competiciones de `silver_achievements`, con 0 partidos.
- Se excluyen las que ya están en `bronze_competiciones`.

In [ ]:
# CELDA 1: configuracion y funciones (correr primero en cada sesion)
import subprocess, re, time, hashlib, os, random, urllib.robotparser
from datetime import datetime
from html import unescape
from urllib.parse import quote, unquote
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

TM = "https://www.transfermarkt.com"
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
CACHE = "/Volumes/workspace/inferiores/scraper/cache"
DELAY = 5.0   # segundos entre paginas (bajar a 3 solo si TM no frena)
TABLA = "workspace.inferiores.bronze_competiciones"
COLS = ["competition_id", "nombre", "tipo_url", "categoria_tm", "fuente_categoria", "indice", "url_final", "estado", "visitado_en"]
SCHEMA = StructType([StructField(c, StringType()) for c in COLS])
INDICES = ["afrika", "amerika", "asien", "europa", "europaJugend", "fifa"]

os.makedirs(CACHE, exist_ok=True)


def curl(url, out):
    r = subprocess.run(["curl", "-sL", "--max-time", "40", "-A", UA, "-w", "%{http_code} %{url_effective}", "-o", out, url],
                       capture_output=True, text=True, timeout=60)
    p = r.stdout.strip().split(" ", 1)
    return p[0], (p[1] if len(p) > 1 else "")


curl(TM + "/robots.txt", "/tmp/robots.txt")
rp = urllib.robotparser.RobotFileParser()
rp.parse(open("/tmp/robots.txt", encoding="utf-8", errors="replace").read().splitlines())


def get(url):
    """Devuelve (codigo, html, url_final). El cache guarda el html y, al lado, la URL final (.url)."""
    base = f"{CACHE}/{hashlib.md5(url.encode()).hexdigest()}"
    if os.path.exists(base + ".html"):
        fin = open(base + ".url", encoding="utf-8").read() if os.path.exists(base + ".url") else url
        return "200", open(base + ".html", encoding="utf-8", errors="replace").read(), fin
    if not rp.can_fetch(UA, url):
        raise RuntimeError(f"robots.txt prohibe: {url}")
    time.sleep(DELAY)
    code, fin, freno = "", "", 0
    for attempt in range(6):
        code, fin = curl(url, base + ".tmp")
        if code == "200":
            os.replace(base + ".tmp", base + ".html")
            open(base + ".url", "w", encoding="utf-8").write(fin)
            return code, open(base + ".html", encoding="utf-8", errors="replace").read(), fin
        if code in ("202", "403", "429"):
            freno += 1
            if freno > 2:
                raise RuntimeError(f"HTTP {code} en {url}: freno")
            print(f"  HTTP {code}: espero 120 s y reintento ({freno}/2)")
            time.sleep(120)
            continue
        if code == "404":
            return code, "", fin
        time.sleep(5 * (attempt + 1))
    return code or "0", "", ""


def limpiar(h):
    s = re.sub(r"(?s)<(script|style)[^>]*>.*?</\1>", " ", h)
    s = re.sub(r"(?s)<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", unescape(s)).strip()


def tabla_items(h):
    """Devuelve el HTML de la tabla `items` (la de las competiciones), respetando tablas anidadas."""
    m = re.search(r'<table[^>]*class="[^"]*\bitems\b[^"]*"[^>]*>', h)
    if not m:
        return ""
    i, depth = m.start(), 0
    for t in re.finditer(r"<table\b|</table>", h[i:]):
        depth += 1 if t.group(0).startswith("<table") else -1
        if depth == 0:
            return h[i:i + t.end()]
    return h[i:]


TOK_IDX = re.compile(r'(?s)<td[^>]*class="[^"]*extrarow[^"]*"[^>]*>(?P<sub>.*?)</td>'
                     r'|href="/[^"]*/startseite/wettbewerb/(?P<id>[^"/?#]+)"')


def leer_indices():
    """{competition_id: (indice, subtitulo)} leyendo solo la tabla de cada pagina de cada indice."""
    res, conflictos, sin_tabla = {}, 0, 0
    for ix in INDICES:
        _, h1, _ = get(f"{TM}/wettbewerbe/{ix}")
        maxp = max([1] + [int(x) for x in re.findall(r"\?page=(\d+)", h1)])
        for p in range(1, maxp + 1):
            h = h1 if p == 1 else get(f"{TM}/wettbewerbe/{ix}?page={p}")[1]
            t = tabla_items(h)
            if not t:
                sin_tabla += 1
                continue
            sub = None
            for m in TOK_IDX.finditer(t):
                if m.group("sub") is not None:
                    tx = limpiar(m.group("sub"))
                    if tx:
                        sub = tx[:60]
                elif m.group("id") and sub:
                    cid = unquote(m.group("id"))
                    if cid not in res:
                        res[cid] = (ix, sub)
                    elif res[cid][1] != sub:
                        conflictos += 1
    print(f"indices: {len(res)} competiciones en las tablas | paginas sin tabla: {sin_tabla} | ids con subtitulo distinto en dos lugares: {conflictos}")
    return res


def leer_pagina(html, final):
    t = re.search(r"(?s)<title>(.*?)</title>", html)
    titulo = unescape(t.group(1)).replace("| Transfermarkt", "").strip() if t else None
    tipo = "pokalwettbewerb" if "/pokalwettbewerb/" in final else "wettbewerb"
    m = re.search(r"League level:\s*(.{0,45})", limpiar(html))
    nivel = re.split(r"\s(?:Reigning|Record|Number|UEFA|€|Total|Most|-)\s?", m.group(1))[0].strip() if m else None
    return titulo, tipo, (nivel or None)


def ahora():
    return datetime.now().isoformat(timespec="seconds")


spark.sql(f"""CREATE TABLE IF NOT EXISTS {TABLA}
              (competition_id STRING, nombre STRING, tipo_url STRING, categoria_tm STRING,
               fuente_categoria STRING, indice STRING, url_final STRING, estado STRING, visitado_en STRING)""")


def guardar(rows):
    if rows:
        spark.createDataFrame(rows, SCHEMA).write.mode("append").saveAsTable(TABLA)


def pendientes():
    """Competiciones de silver_player_stats (con partidos) y de silver_achievements que todavia no estan en la tabla."""
    s = spark.sql("""SELECT competition_id, SUM(partidos) AS partidos, MAX(nombre) AS nombre FROM (
                       SELECT competition_id, SUM(appearances) AS partidos, MAX(competition_name) AS nombre
                       FROM workspace.inferiores.silver_player_stats
                       WHERE appearances > 0 AND competition_id IS NOT NULL GROUP BY competition_id
                       UNION ALL
                       SELECT competition_id, 0 AS partidos, MAX(competition_name) AS nombre
                       FROM workspace.inferiores.silver_achievements
                       WHERE competition_id IS NOT NULL GROUP BY competition_id
                     ) GROUP BY competition_id""")
    hechos = spark.table(TABLA).select("competition_id")
    return s.join(hechos, "competition_id", "left_anti").orderBy(F.desc("partidos")).collect()


print("listo: TABLA =", TABLA, "| DELAY =", DELAY)

## CELDA 3: rondas
Se puede cortar y volver a correr: retoma donde quedó.

| Parámetro | Valor por defecto | Para qué sirve |
|---|---|---|
| `MAX_VISITAS` | `None` | Máximo de páginas a visitar en la corrida. Sin valor, no hay límite |
| `MAX_MIN` | `None` | Minutos máximos de la corrida. Sin valor, no hay límite |

**Pasos**

1. `leer_indices()` arma el mapa de competiciones de los índices. Imprime la cantidad de pendientes, cuántas tienen subtítulo de índice (no se visitan), cuántas hay que visitar y una estimación de minutos.
2. Las pendientes que están en un índice se guardan sin visitar: `categoria_tm` es el subtítulo, `fuente_categoria` es `indice`, `indice` es la región y `estado` es `indice`.
3. Las demás se visitan una por una en `/x/startseite/wettbewerb/{id}` y se guardan según el resultado:

| Resultado de la visita | Qué se guarda |
|---|---|
| La página no devuelve contenido | `estado = sin_pagina`, sin categoría |
| La página trae `League level` | `estado = ok`, `fuente_categoria = pagina` y `categoria_tm` con el nivel |
| La página no trae nivel | `estado = sin_nivel`, sin categoría |

4. Escribe cada 25 filas y al final. Si hay freno o `robots.txt` prohíbe una ruta, imprime `CORTE`, guarda el búfer y termina.

In [ ]:
# CELDA 3: RONDAS. Se puede cortar (MAX_VISITAS / MAX_MIN) y volver a correr: retoma donde quedo.
MAX_VISITAS = None   # por ejemplo 100 para una tanda corta
MAX_MIN = None       # por ejemplo 30 para cortar a los 30 minutos

t0, visitas, buf = time.time(), 0, []
idx = leer_indices()
pend = pendientes()
directos = [r for r in pend if r.competition_id in idx]
a_visitar = [r for r in pend if r.competition_id not in idx]
print(f"pendientes: {len(pend)} | con subtitulo del indice (sin visitar): {len(directos)} | a visitar: {len(a_visitar)} "
      f"(~{len(a_visitar) * (DELAY + 1) / 60:.0f} min)")

guardar([(r.competition_id, r.nombre, None, idx[r.competition_id][1], "indice", idx[r.competition_id][0],
          None, "indice", ahora()) for r in directos])

try:
    for r in a_visitar:
        if MAX_VISITAS and visitas >= MAX_VISITAS:
            break
        if MAX_MIN and (time.time() - t0) / 60 >= MAX_MIN:
            break
        cid = r.competition_id
        code, html, fin = get(f"{TM}/x/startseite/wettbewerb/{quote(cid, safe='')}")
        if not html:
            buf.append((cid, r.nombre, None, None, None, None, fin or None, "sin_pagina", ahora()))
        else:
            titulo, tipo, nivel = leer_pagina(html, fin)
            buf.append((cid, titulo or r.nombre, tipo, nivel, "pagina" if nivel else None, None, fin,
                        "ok" if nivel else "sin_nivel", ahora()))
        visitas += 1
        if len(buf) >= 25:
            guardar(buf)
            buf = []
            print(f"  {visitas}/{len(a_visitar)} visitas, {(time.time() - t0) / 60:.1f} min")
except RuntimeError as e:
    print("CORTE:", e)
finally:
    guardar(buf)
    print(f"guardadas {visitas} visitas en esta corrida")

## CELDA 4: resumen
Imprime cuántas competiciones siguen pendientes y muestra tres consultas sobre `bronze_competiciones`:

1. Cantidad de competiciones por `fuente_categoria`, `estado` y `tipo_url`.
2. Cantidad de competiciones y de partidos por categoría de Transfermarkt (solo competiciones con partidos en `silver_player_stats`). Las que no tienen categoría figuran como `(sin categoria)`.
3. Las 20 competiciones sin categoría con más partidos.

In [ ]:
# CELDA 4: RESUMEN
pend = pendientes()
print(f"Pendientes: {len(pend)} competiciones")

print("\nPor fuente y estado:")
display(spark.sql(f"""
SELECT fuente_categoria, estado, tipo_url, COUNT(*) AS competiciones
FROM {TABLA} GROUP BY fuente_categoria, estado, tipo_url ORDER BY competiciones DESC"""))

print("\nPor categoria de Transfermarkt (con partidos):")
display(spark.sql(f"""
WITH p AS (SELECT competition_id, SUM(appearances) AS partidos
           FROM workspace.inferiores.silver_player_stats WHERE appearances > 0 GROUP BY competition_id)
SELECT COALESCE(b.categoria_tm, '(sin categoria)') AS categoria, COUNT(*) AS competiciones, SUM(p.partidos) AS partidos
FROM {TABLA} b JOIN p ON p.competition_id = b.competition_id
GROUP BY 1 ORDER BY partidos DESC"""))

print("\nLas 20 con mas partidos que quedaron sin categoria (para la regla de paises en gold):")
display(spark.sql(f"""
SELECT b.competition_id, b.nombre, b.tipo_url, b.estado, SUM(s.appearances) AS partidos
FROM {TABLA} b JOIN workspace.inferiores.silver_player_stats s ON s.competition_id = b.competition_id
WHERE b.categoria_tm IS NULL AND s.appearances > 0
GROUP BY b.competition_id, b.nombre, b.tipo_url, b.estado ORDER BY partidos DESC LIMIT 20"""))